In [29]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [30]:
df = pd.read_csv(r"C:\Users\victo\Downloads\spy_2020_2022.csv", low_memory=False)

In [31]:
df.columns = df.columns.str.strip().str.strip('[]')
df = df.drop(['QUOTE_UNIXTIME', 'QUOTE_READTIME', 'QUOTE_TIME_HOURS', 'EXPIRE_UNIX', 'C_DELTA', 'C_GAMMA', 'C_VEGA', 'C_THETA', 'C_RHO', 'P_DELTA', 'P_GAMMA', 'P_VEGA', 'P_THETA', 'P_RHO'], axis=1)
df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)
df['EXPIRE_DATE'] = pd.to_datetime(df['EXPIRE_DATE'], errors='coerce')
df['QUOTE_DATE'] = pd.to_datetime(df['QUOTE_DATE'], errors='coerce')
date_cols = ['EXPIRE_DATE', 'QUOTE_DATE']
cols_to_exclude = ['EXPIRE_DATE', 'QUOTE_DATE', 'C_SIZE', 'P_SIZE']
num_cols = df.columns.difference(cols_to_exclude)
df[date_cols] = df[date_cols].apply(pd.to_datetime, format='%Y-%m-%d', errors='coerce')
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df.head(n=10)

C:\Users\victo\AppData\Local\Temp\ipykernel_9412\1280743297.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)


,QUOTE_DATE,UNDERLYING_LAST,EXPIRE_DATE,DTE,C_IV,C_VOLUME,C_LAST,C_SIZE,C_BID,C_ASK,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,STRIKE_DISTANCE,STRIKE_DISTANCE_PCT
0,2021-09-01,451.85,2021-09-01,0.0,NaN,1.0,182.65,1 x 1,181.09,182.31,270.0,0.0,0.01,0 x 2239,0.01,3.41249,3.0,181.9,0.402
1,2021-09-01,451.85,2021-09-01,0.0,NaN,NaN,0.00,1 x 1,176.09,177.31,275.0,0.0,0.01,0 x 2679,0.01,3.29737,0.0,176.9,0.391
2,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,157.75,1 x 1,171.09,172.31,280.0,0.0,0.01,0 x 2679,0.01,3.18330,11.0,171.9,0.380
3,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,153.45,1 x 1,166.14,167.30,285.0,0.0,0.01,0 x 2679,0.01,3.07217,50.0,166.9,0.369
4,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,147.76,1 x 1,161.09,162.31,290.0,0.0,0.01,0 x 2679,0.01,2.96230,0.0,161.9,0.358
5,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,143.46,200 x 200,156.10,157.30,295.0,0.0,0.01,0 x 2679,0.04,2.85433,1.0,156.9,0.347
6,2021-09-01,451.85,2021-09-01,0.0,NaN,24.0,138.51,200 x 200,151.10,152.30,300.0,0.0,0.01,0 x 2679,0.01,2.74842,3.0,151.9,0.336
7,2021-09-01,451.85,2021-09-01,0.0,NaN,24.0,134.11,1 x 2,146.15,147.31,305.0,0.0,0.01,0 x 2679,0.02,2.64392,0.0,146.9,0.325
8,2021-09-01,451.85,2021-09-01,0.0,NaN,NaN,0.00,200 x 200,141.10,142.30,310.0,0.0,0.01,0 x 2174,0.08,2.54076,1.0,141.9,0.314
9,2021-09-01,451.85,2021-09-01,0.0,NaN,10.0,137.09,200 x 200,136.10,137.30,315.0,0.0,0.01,0 x 2679,0.00,2.43886,NaN,136.9,0.303


In [32]:
df = df.drop(['C_IV', 'C_VOLUME', 'C_LAST', 'C_SIZE', 'C_BID', 'C_ASK', 'STRIKE_DISTANCE', 'STRIKE_DISTANCE_PCT', 'EXPIRE_DATE'], axis=1)
df['moneyness'] = df['UNDERLYING_LAST'] / df['STRIKE']
df = df[df['P_BID'] > 0]  # filter out 0 bid options
df['midprice'] = (df['P_BID'] + df['P_ASK']) / 2.0
df = df[df['DTE'] > 0]  # filter out 0DTE options
df = df[df['P_VOLUME'] > 0] # filter out 0 volume options
df = df[df['P_IV'] < 3] # filter out extreme IV values
df = df[df['P_IV'] > 0] # filter out extreme IV values
df['T'] = df['DTE'] / 365.0
df.head(n=10)

,QUOTE_DATE,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T
157,2021-09-01,451.85,2.0,397.0,0.01,0.02,2040 x 4188,0.01,0.53525,452.0,1.138161,0.015,0.005479
158,2021-09-01,451.85,2.0,398.0,0.01,0.02,2359 x 4347,0.01,0.52504,11.0,1.135302,0.015,0.005479
160,2021-09-01,451.85,2.0,400.0,0.01,0.02,2326 x 3288,0.01,0.50651,20.0,1.129625,0.015,0.005479
162,2021-09-01,451.85,2.0,402.0,0.01,0.02,2286 x 2453,0.02,0.48794,2.0,1.124005,0.015,0.005479
164,2021-09-01,451.85,2.0,404.0,0.01,0.02,2246 x 2413,0.02,0.47024,3.0,1.118441,0.015,0.005479
165,2021-09-01,451.85,2.0,405.0,0.01,0.02,2226 x 3926,0.01,0.46041,55.0,1.115679,0.015,0.005479
167,2021-09-01,451.85,2.0,407.0,0.01,0.02,2186 x 3223,0.02,0.43999,1.0,1.110197,0.015,0.005479
168,2021-09-01,451.85,2.0,408.0,0.01,0.02,2171 x 3059,0.02,0.43037,2.0,1.107475,0.015,0.005479
169,2021-09-01,451.85,2.0,409.0,0.01,0.02,2162 x 3437,0.02,0.42194,1.0,1.104768,0.015,0.005479
170,2021-09-01,451.85,2.0,410.0,0.01,0.02,3092 x 3486,0.02,0.41266,13.0,1.102073,0.015,0.005479


In [33]:
df['log_moneyness'] = np.log(df['moneyness'])
df['q'] = 0.015  # constant dividend yield
df.head(n=10)

,QUOTE_DATE,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q
157,2021-09-01,451.85,2.0,397.0,0.01,0.02,2040 x 4188,0.01,0.53525,452.0,1.138161,0.015,0.005479,0.129414,0.015
158,2021-09-01,451.85,2.0,398.0,0.01,0.02,2359 x 4347,0.01,0.52504,11.0,1.135302,0.015,0.005479,0.126898,0.015
160,2021-09-01,451.85,2.0,400.0,0.01,0.02,2326 x 3288,0.01,0.50651,20.0,1.129625,0.015,0.005479,0.121886,0.015
162,2021-09-01,451.85,2.0,402.0,0.01,0.02,2286 x 2453,0.02,0.48794,2.0,1.124005,0.015,0.005479,0.116898,0.015
164,2021-09-01,451.85,2.0,404.0,0.01,0.02,2246 x 2413,0.02,0.47024,3.0,1.118441,0.015,0.005479,0.111935,0.015
165,2021-09-01,451.85,2.0,405.0,0.01,0.02,2226 x 3926,0.01,0.46041,55.0,1.115679,0.015,0.005479,0.109463,0.015
167,2021-09-01,451.85,2.0,407.0,0.01,0.02,2186 x 3223,0.02,0.43999,1.0,1.110197,0.015,0.005479,0.104537,0.015
168,2021-09-01,451.85,2.0,408.0,0.01,0.02,2171 x 3059,0.02,0.43037,2.0,1.107475,0.015,0.005479,0.102083,0.015
169,2021-09-01,451.85,2.0,409.0,0.01,0.02,2162 x 3437,0.02,0.42194,1.0,1.104768,0.015,0.005479,0.099635,0.015
170,2021-09-01,451.85,2.0,410.0,0.01,0.02,3092 x 3486,0.02,0.41266,13.0,1.102073,0.015,0.005479,0.097193,0.015


In [34]:
rf = pd.read_csv(r"C:\Users\victo\Downloads\DTB3.csv", low_memory=False)
rf['observation_date'] = pd.to_datetime(rf['observation_date'], format='%Y-%m-%d', errors='coerce')
rf.rename(columns={'observation_date': 'date', 'DTB3': 'risk_free_rate'}, inplace=True)
rf['risk_free_rate'] /= 100
rf.sort_values('date', inplace=True)
rf.fillna(method='ffill', inplace=True)
rf.head(n=10)

C:\Users\victo\AppData\Local\Temp\ipykernel_9412\2741189155.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  rf.fillna(method='ffill', inplace=True)


,date,risk_free_rate
0,2015-05-08,0.0001
1,2015-05-11,0.0002
2,2015-05-12,0.0003
3,2015-05-13,0.0002
4,2015-05-14,0.0001
5,2015-05-15,0.0002
6,2015-05-18,0.0002
7,2015-05-19,0.0002
8,2015-05-20,0.0002
9,2015-05-21,0.0002


In [ ]:
# Merge on the 'date' column
df.rename(columns={'QUOTE_DATE': 'date'}, inplace=True)
df = df.merge(rf, on='date', how='left')

In [38]:
df.head(n=10)

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q,risk_free_rate
0,2021-09-01,451.85,2.0,397.0,0.01,0.02,2040 x 4188,0.01,0.53525,452.0,1.138161,0.015,0.005479,0.129414,0.015,0.0005
1,2021-09-01,451.85,2.0,398.0,0.01,0.02,2359 x 4347,0.01,0.52504,11.0,1.135302,0.015,0.005479,0.126898,0.015,0.0005
2,2021-09-01,451.85,2.0,400.0,0.01,0.02,2326 x 3288,0.01,0.50651,20.0,1.129625,0.015,0.005479,0.121886,0.015,0.0005
3,2021-09-01,451.85,2.0,402.0,0.01,0.02,2286 x 2453,0.02,0.48794,2.0,1.124005,0.015,0.005479,0.116898,0.015,0.0005
4,2021-09-01,451.85,2.0,404.0,0.01,0.02,2246 x 2413,0.02,0.47024,3.0,1.118441,0.015,0.005479,0.111935,0.015,0.0005
5,2021-09-01,451.85,2.0,405.0,0.01,0.02,2226 x 3926,0.01,0.46041,55.0,1.115679,0.015,0.005479,0.109463,0.015,0.0005
6,2021-09-01,451.85,2.0,407.0,0.01,0.02,2186 x 3223,0.02,0.43999,1.0,1.110197,0.015,0.005479,0.104537,0.015,0.0005
7,2021-09-01,451.85,2.0,408.0,0.01,0.02,2171 x 3059,0.02,0.43037,2.0,1.107475,0.015,0.005479,0.102083,0.015,0.0005
8,2021-09-01,451.85,2.0,409.0,0.01,0.02,2162 x 3437,0.02,0.42194,1.0,1.104768,0.015,0.005479,0.099635,0.015,0.0005
9,2021-09-01,451.85,2.0,410.0,0.01,0.02,3092 x 3486,0.02,0.41266,13.0,1.102073,0.015,0.005479,0.097193,0.015,0.0005
